# Notebook 4: Data Preprocessing for Topic Modelling

## Purpose
Prepare the **NEGATIVE** sentiment comments for topic modelling using a **different** stopword list from Notebook 2.

## Why different stopwords?
- **Sentiment analysis** needs opinion words like "hate", "love", "terrible"
- **Topic modelling** needs domain-specific nouns/verbs to form coherent clusters
- So here we ALSO remove fast-food generics ("order", "food", "restaurant") that would dominate every topic

## Pipeline
1. Load fully labelled dataset → filter to Negative only
2. Apply topic-specific stopword list
3. Clean, tokenise, lemmatise
4. Generate word cloud of negative comments
5. Save preprocessed subset

## Input
`kfc_sentiment_labelled_full.csv` (from Notebook 3)

## Output
- `kfc_negative_comments.csv`
- `wordcloud_negative_comments.png`

## Step 0 — Import Libraries

In [ ]:
# !pip install pandas nltk wordcloud matplotlib

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
print("Libraries loaded.")

## Step 1 — Configuration

In [ ]:
INPUT_FILE     = "kfc_sentiment_labelled_full.csv"
OUTPUT_FILE    = "kfc_negative_comments.csv"
WORDCLOUD_FILE = "wordcloud_negative_comments.png" 

## Step 2 — Topic Modelling Stopwords

This list is **different** from Notebook 2. We now also remove fast-food-generic terms  
that would appear in every topic and reduce coherence.

| Added removals | Why |
|---|---|
| drive, thru, restaurant, meal, staff, location | Appear in every complaint — no topic distinction |
| food, eat, menu, service, customer, place | Too generic for topic separation |

In [ ]:
TOPIC_STOPWORDS = set(stopwords.words("english")).union({
    # Brand-specific
    "kfc", "kentucky", "fried",
    # Fast food generic (would dominate every topic)
    "drive", "thru", "fast", "food", "order", "ordered", "restaurant",
    "meal", "staff", "location", "customer", "service", "place", "menu", "eat", "eating", "ate",
    # Common verbs
    "got", "get", "could", "also", "like", "do", "does", "did", "have", "has",
    "having", "be", "was", "been", "had", "make", "go", "going", "went", "come", "came",
    # Contractions / Informal
    "im", "dont", "didnt", "theyll", "youre", "ive", "theyre", "thats",
    "yeah", "lol", "u", "us", "cant", "wont", "isnt",
    # Time / Reference
    "time", "now", "then", "there", "when", "day", "year", "today",
    "always", "ago", "last", "still", "just", "even",
    # Pronouns
    "they", "them", "you", "me", "my", "he", "she", "we", "him",
    "her", "our", "ours", "yours", "theirs", "their", "i",
    # Articles / Determiners / Prepositions / Conjunctions
    "the", "a", "an", "this", "that", "these", "those", "its",
    "to", "of", "in", "on", "at", "from", "with", "about", "for",
    "and", "but", "or", "if", "as", "so", "because", "though",
    # Question words / Auxiliaries
    "what", "which", "who", "whom", "whose", "how", "where", "why",
    "is", "are", "am", "was", "were", "will", "would", "can", "could", "should", "shall",
    # Degree / Misc
    "really", "much", "actually", "every", "everything",
    "one", "thing", "not", "people", "said", "say", "way",
    "something", "back", "new", "know", "think", "see", "need", "want", "used", "use", "work",
})

print(f"Topic stopwords: {len(TOPIC_STOPWORDS)} words")

## Step 3 — Load and Filter to Negative Only

In [ ]:
df = pd.read_csv(INPUT_FILE)
print(f"Full dataset: {len(df)} comments")
print(f"\nSentiment breakdown:")
print(df["Sentiment"].value_counts())

df_neg = df[df["Sentiment"] == "Negative"].copy()
print(f"\nNegative comments extracted: {len(df_neg)}")

## Step 4 — Preprocess for Topic Modelling

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"/[ur]/\w+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_for_topics(text):
    """Clean → tokenise → remove topic stopwords → lemmatise."""
    lemmatizer = WordNetLemmatizer()
    cleaned = clean_text(text)
    tokens = cleaned.split()
    tokens = [t for t in tokens if t not in TOPIC_STOPWORDS and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

print("Preprocessing negative comments...")
df_neg["Tokenised_Comment"] = df_neg["Comment"].apply(preprocess_for_topics)

before = len(df_neg)
df_neg = df_neg[df_neg["Tokenised_Comment"].str.strip().astype(bool)]
after = len(df_neg)

print(f"Removed {before - after} empty rows")
print(f"Final negative comments: {after}")

## Step 5 — Word Cloud of Negative Comments

In [ ]:
all_neg_words = " ".join(df_neg["Tokenised_Comment"])
wordcloud = WordCloud(
    width=1200, height=600, background_color="white",
    max_words=200, colormap="Reds",
).generate(all_neg_words)

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation="bilinear")
plt.title("Word Cloud of Negative Comments (KFC — BERTweet Filtered)", fontsize=16)
plt.axis("off")
plt.tight_layout()
plt.savefig(WORDCLOUD_FILE, dpi=150, bbox_inches="tight")
plt.show()

## Step 6 — Save Preprocessed Negative Dataset

In [ ]:
df_neg.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"Saved to: {OUTPUT_FILE}")
print(f"Negative comments: {len(df_neg)}")
print(f"\nSample tokens:")
print(df_neg["Tokenised_Comment"].iloc[0])